# 示例: 3. 单位线汇流示例

**脚本:** `examples/run_uh_example.py`

## 目的
演示 `UnitHydrographRouting` 模块的用法。脚本定义了一个单位线（UH），然后将一系列有效降雨脉冲输入模块。模块通过卷积运算，计算出总的直接径流过程线，并用图表展示结果。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.routing import UnitHydrographRouting

## 2. 定义单位线

单位线（Unit Hydrograph, UH）是单位时段内单位净雨在流域出口断面形成的地面径流过程线。这里我们定义一个简单的单位线，其纵坐标代表了不同时刻的流量。注意，理论上单位线的所有纵坐标之和应该为1（或接近1）。

In [ ]:
uh_ordinates = np.array([0.1, 0.3, 0.4, 0.15, 0.05])

## 3. 创建汇流模块实例

我们使用上面定义的单位线来创建一个`UnitHydrographRouting`的实例。

In [ ]:
uh_router = UnitHydrographRouting(uh_ordinates=uh_ordinates)

## 4. 创建有效降雨序列

我们创建一个有效降雨（净雨）序列，这通常是总降雨减去各种损失（如入渗、蒸发）后的结果。在实际应用中，这部分雨量会由产流模块计算得出。

In [ ]:
effective_rainfall = np.array([10, 25, 5, 0, 0, 0, 0, 0, 0, 0])
timesteps = np.arange(len(effective_rainfall))

print(f"输入有效降雨 (mm): {effective_rainfall}")

## 5. 运行汇流模拟

单位线法的核心是卷积运算。模块在内部将输入的有效降雨序列与单位线进行卷积，从而得到总的直接径流过程线。

In [ ]:
outflow_hydrograph = []
for rain_val in effective_rainfall:
    outflow_val = uh_router.run(rain_val)
    outflow_hydrograph.append(outflow_val)

outflow_hydrograph = np.array(outflow_hydrograph)

print(f"输出直接径流过程线 (mm): {np.round(outflow_hydrograph, 2)}")

## 6. 可视化结果

最后，我们绘制有效降雨和计算出的径流过程线。可以看到，多场次的降雨产生的径流过程线被正确地叠加在一起，形成了总的出流过程。

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# 绘制降雨
ax1.bar(timesteps, effective_rainfall, width=0.8, color='c', alpha=0.6, label='Effective Rainfall')
ax1.set_xlabel('Time Step (days)')
ax1.set_ylabel('Rainfall (mm)', color='c')
ax1.tick_params(axis='y', labelcolor='c')
ax1.invert_yaxis()

# 在第二个y轴上绘制过程线
ax2 = ax1.twinx()
ax2.plot(timesteps, outflow_hydrograph, 'r-o', label='Direct Runoff Hydrograph')
ax2.set_ylabel('Flow (mm depth equivalent)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Unit Hydrograph Routing Example')
fig.tight_layout()

output_dir = '../examples/results/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'uh_example_plot.png')
plt.savefig(output_path)
print(f"\n汇流图已保存到 {output_path}")
plt.show()